In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
pd.set_option("display.max_columns", None)

In [3]:
import pandas as pd
import numpy as np
import json
import random
from src.utils import df_features_to_numpy, df_target_to_numpy, run_experiment_pytorch_clasificacion, run_experiment_pytorch_regresion, permutation_importance_pytorch
from sklearn.preprocessing import StandardScaler
from src.splitting import split_estratificado_autor

In [4]:
#otra manera de leer el dataset, introduce aleatoriedad en las filas, no agarra las 100mil primeras
#tardo aprox 3min14s en correr
K = 100_000
sample = []

with open("goodreads_us_english_primer_filtrado.json", encoding="utf8") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)

        if i < K:
            sample.append(obj)
        else:
            j = random.randint(0, i)
            if j < K:
                sample[j] = obj

df = pd.DataFrame(sample)
#REPETIR PARA EL TRAIN, VAL Y TEST
#TODO: comentario de clara para clara: descomentar este bloque antes de correr

## Temporal, sacar cuando este el json ya procesado posta

In [5]:
#TEMPORAL: normalmente estas features ya van a estar agrupadas
from src.features import normalize_format
#Le aplico la funcion para reagrupar:
if "format_group" not in df.columns:
    df["format_group"] = df["format"].apply(normalize_format)

print(df["format_group"].value_counts())

format_group
PAPERBACK    53861
HARDCOVER    22361
EBOOK        17145
UNKNOWN       4393
AUDIO         1638
OTHER          554
COMIC           48
Name: count, dtype: int64


In [6]:
#TEMPORAL: normalmente los generos ya van a estar definidos
from src.features import extraer_generos_libro
df["genres"] = df["popular_shelves"].apply(extraer_generos_libro)

In [7]:
#TEMPORAL: ya va a estar hecho el one hot de estas features
from src.features import multi_binarizer_format, multi_binarizer_genre, one_hot_format
#df = multi_binarizer_format(df) #multi binarizer de format
df = one_hot_format(df)
df = multi_binarizer_genre(df) #multi binarizer de genero

In [8]:
def extraer_autor_principal(autores_sin_preprocesar):
    """
    autores_sin preprocesar: [{authorId, role}, {authorid, role})]
    Asumimos que el primer elemento de la lista es el autor principal (orden tipico de Goodreads)
    """
    # Buscar el primero con role vacío; si no hay, usar el primero de la lista
    for dic_autores in autores_sin_preprocesar:
        if dic_autores["role"] == "":
            return dic_autores["author_id"]
    return autores_sin_preprocesar[0]["author_id"]

df = df[df["authors"].apply(len) > 0]
df["author_id_principal"] = df["authors"].apply(extraer_autor_principal)

In [9]:
df_train, df_val, df_test = split_estratificado_autor(df, test_size=0.15, val_size=0.15)

print(f"Train: {len(df_train)} ({len(df_train)/len(df):.1%})")
print(f"Val: {len(df_val)} ({len(df_val)/len(df):.1%})")
print(f"Test: {len(df_test)} ({len(df_test)/len(df):.1%})")

#Verificacion: distribución de estrellas en cada split
for nombre, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"\n{nombre}:")
    print(d["estrellas"].value_counts(normalize=True).sort_index())

Train: 72171 (72.2%)
Val: 12563 (12.6%)
Test: 15266 (15.3%)

train:
estrellas
0    0.002605
1    0.001316
2    0.008702
3    0.146929
4    0.775741
5    0.064707
Name: proportion, dtype: float64

val:
estrellas
0    0.002627
1    0.001433
2    0.008915
3    0.144233
4    0.778317
5    0.064475
Name: proportion, dtype: float64

test:
estrellas
0    0.002620
1    0.001245
2    0.008254
3    0.143653
4    0.779641
5    0.064588
Name: proportion, dtype: float64


# FIN DE TEMPORAL

In [10]:
generos = ["fiction", "non fiction", "fantasy", "science fiction", "mystery", "thriller", "romance", "horror", "historical", "young adult", "children", "adventure", "contemporary", "dystopian", "paranormal", "urban", "crime", "biography", "memoir", "self help", "history", "psychology", "poetry", "classics", "graphic novel", "short stories", "humor", "religion"]
features = ["is_series", "publication_year", "num_pages", 'format_group_AUDIO', 'format_group_COMIC', 'format_group_EBOOK', 'format_group_HARDCOVER', 'format_group_OTHER', 'format_group_PAPERBACK'] + generos #'format_group_UNKNOWN' la saco por el one hot en regresion

#Me quedo solo con las features necesarias para el modelo base y las estandarizo
dfs = [df_train, df_val, df_test]
y_train_regresion, y_val_regresion, y_test_regresion = df_target_to_numpy("average_rating", dfs)
y_train_clasificacion, y_val_clasificacion, y_test_clasificacion = df_target_to_numpy("estrellas", dfs)
X_train, X_val, X_test = df_features_to_numpy(features, dfs)

#estandarizo los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val) #escala X_val con los valores sacados de X_train
#TODO: escalar el X_test

In [ ]:
from sklearn.neighbors import NearestNeighbors

modelo = NearestNeighbors(n_neighbors = 20, metric = "cosine")
modelo.fit(X_train_scaled)

def recomendar(modelo, X, indice_libro, n_recomendaciones = 10):
    distancias, indices = modelo.kneighbors(X[indice_libro].reshape(1, -1), n_neighbors = n_recomendaciones + 1)
    return indices[0][1:], distancias[0][1:]

In [16]:
idx = df_train[df_train["title"] == "The Book Thief"].index[0]
print(idx)

6016


TODO: que la persona pueda poner tb el autor, porq un mismo titulo puede correpsonder a diferentes libros

Va a funcionar mal porq todavia no puse los embeddings, pero asi seria aprox el codigo :)

In [17]:
recomendados, distancias = recomendar(modelo, X_train_scaled, 6016, 10)

print("\nRecomendaciones:")
for idx, dist in zip(recomendados, distancias):
    print(f"{df_train.iloc[idx]['title']}  (distancia = {dist:.4f})")


Recomendaciones:
Refugee (Bio of a Space Tyrant, #1)  (distancia = 0.1565)
Demon (Gaea, #3)  (distancia = 0.1654)
Steel Gauntlet (Starfist, #3)  (distancia = 0.1753)
Doctor Who: Relative Dementias  (distancia = 0.1880)
Courageous (The Lost Fleet, #3)  (distancia = 0.2055)
Life Probe  (distancia = 0.2153)
Mere Mortals (Star Trek: Destiny #2)  (distancia = 0.2199)
Courageous (The Lost Fleet, #3)  (distancia = 0.2240)
Transcendence (Heritage Universe 3)  (distancia = 0.2241)
Empire of Blood (Richard Blade, #23)  (distancia = 0.2244)
